In [1]:
import numpy as np
from matplotlib import pyplot as plt
import matplotlib as mpl
import ipywidgets as widgets
from IPython.display import display
from ipyfilechooser import FileChooser
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
import pygwalker as pyg
import polars as pl
from datetime import datetime
from pathlib import Path
import yfinance as yf
pio.renderers.default='notebook' # for vscode ,maybe 'colab' on jupyterlab 
plt.ioff();

In [2]:
def init_fig():
    global fig
    fig = plt.figure()
    plt.close()
    

def add_scatter_xy():
    global fig
    global marker, marker_size
    global color
    global x,y
    plt.get_current_fig_manager().canvas.figure = fig
    plt.scatter(x,y, marker=marker, s=marker_size, color=color)
    plt.show()

def add_scatter_y():
    global fig
    global marker, marker_size
    global color
    global y
    plt.get_current_fig_manager().canvas.figure = fig
    x = (np.ones_like(y.T)*range(y.shape[0])).T
    plt.scatter(x,y, marker=marker, s=marker_size, color=color)
    plt.show()

def add_line_xy():
    global fig
    global marker, marker_size
    global linewidth
    global color
    global x, y
    plt.get_current_fig_manager().canvas.figure = fig
    plt.plot(x, y, marker=marker, markersize = marker_size, linewidth=linewidth, color=color)
    plt.show()

def add_line_y():
    global fig
    global marker, marker_size
    global linewidth
    global color
    global y
    plt.get_current_fig_manager().canvas.figure = fig
    plt.plot(y, marker=marker, marker_size=marker_size, linewidth=linewidth, color=color)
    plt.show()

def show_fig():
    global fig
    plt.get_current_fig_manager().canvas.figure = fig
    plt.show()

def set_label_title():
    global xlabel, ylabel, title
    plt.get_current_fig_manager().canvas.figure = fig
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.show()

def set_axis_lim():
    global fig
    global xlim_high, xlim_low
    global ylim_high, ylim_low
    plt.get_current_fig_manager().canvas.figure = fig
    plt.xlim([xlim_low, xlim_high])
    plt.ylim([ylim_low, ylim_high])
    plt.show()

def set_autoscale():
    global fig
    plt.get_current_fig_manager().canvas.figure = fig
    plt.gca().relim()
    plt.gca().autoscale()
    plt.show()

def save_fig_pdf():
    global fig
    global comment
    plt.get_current_fig_manager().canvas.figure = fig
    plt.savefig(f'{comment}.pdf')
    plt.close()

def save_fig_png():
    global fig
    global comment
    plt.get_current_fig_manager().canvas.figure = fig
    plt.savefig(f'{comment}.png', dpi=300)
    plt.close()


def fetch_yf_query(query: yf.EquityQuery, size=250):
    offset = 0
    all_quotes = []

    while True:
        res = yf.screen(query, size=size, offset=offset)

        count = res.get("count", 0)
        quotes = res.get("quotes", [])

        if count == 0:
            break

        all_quotes.extend(quotes)

        offset += count   # 次のページへ

    return all_quotes


def yf_download(tickers: list[str],period=None,start=None,end=None,interval="1d"):
    assert interval in ["1m","5m","30m","60m","90m","1h","1d","1wk","1mo","1y"]
    #assert (period is not None) or (start is not None and end is not None)
    #assert (period is None) or (period in ["1d","5d","1mo","3mo","6mo","1y","2y","5y","10y","ytd","max"])
    # or start="2020-01-01", end="2020-12-31"

    df_multi = yf.download(tickers, start=start, end=end, period=period,group_by='ticker', interval=interval, progress=False)
    dfs = []
    for ticker in tickers:
        df = pl.from_pandas(df_multi[ticker], include_index=True).with_columns([
            pl.lit(ticker).alias("Ticker"),
            pl.lit(interval).alias("interval")
        ])
        dfs.append(df)
    df = pl.concat(dfs, how="diagonal_relaxed")

    return df

# tickers = ['7203.T', '8411.T']
# yf_download(tickers)




def yf_history(ticker: str,period=None, start=None, end=None, interval="1d"):
    assert interval in ["1m","5m","30m","60m","90m","1h","1d","1wk","1mo","1y"]
    #assert (period is not None) or (start is not None and end is not None)
    #assert (period is None) or (period in ["1d","5d","1mo","3mo","6mo","1y","2y","5y","10y","ytd","max"])
    # or start="2020-01-01", end="2020-12-31"


    company = yf.Ticker(ticker)

    # contain information with dict
    # company.info

    df_company = company.history(start='2021-01-01', end='2026-03-01', interval=interval)

    df = pl.from_pandas(df_company, include_index=True).with_columns([
        pl.lit(ticker).alias("Ticker"),
        pl.lit(interval).alias("interval")
        ])
    
    return df

# ticker = '7203.T'
# yf_download(ticker)


def calc_return(df, on='Open'):
    df = df.sort(["Ticker", "Date"]).with_columns([
        # 日付差分（days）→ 年換算
        (
            (pl.col("Date") - pl.col("Date").shift(1))
            .dt.total_days() / 365.0
        )
        .over("Ticker")
        .alias("_dt"),

        # log価格
        pl.col(on).log().alias(f"log_price"),
    ])

    df = df.with_columns([
        ((pl.col("log_price") - pl.col("log_price").shift(1)) / pl.col("_dt"))
            .over("Ticker").alias("return"),
    ])


    df = df.drop("_dt")
    return df

# df = calc_return(df, on="Open")

def yf_sector(sector):
    assert sector in ['basic-materials','communication-services','consumer-cyclical','consumer-defensive','energy','financial-services','healthcare','industrials','real-estate','technology','utilities']
    sector = yf.Sector(key=sector)

    display(sector.overview)
    display(sector.industries)
    display(sector.research_reports)
    display(sector.symbol)
    display(sector.top_companies)
    display(sector.ticker)
    display(sector.top_etfs)
    display(sector.top_mutual_funds)


def yf_ticker(ticker:str):
    ticker = yf.Ticker(ticker)

    display(ticker.calendar)
    display(ticker.actions)
    display(ticker.analyst_price_targets)
    display(ticker.balance_sheet)
    display(ticker.cash_flow)
    display(ticker.income_stmt)
    display(ticker.earnings_estimate)
    display(ticker.info)
    display(ticker.recommendations)
    display(ticker.major_holders)
    display(ticker.institutional_holders)
    display(ticker.mutualfund_holders)



In [3]:
def simple_ui():
    global filechooser
    state_select = {}

    button_clear_output = widgets.Button(description='表示クリア')
    button_init_fig = widgets.Button(description='初期化')
    button_set_label_title = widgets.Button(description='ラベル・タイトル')
    button_set_axis_lim = widgets.Button(description='軸範囲設定')
    button_set_autoscale = widgets.Button(description='自動範囲設定')
    button_add_line_xy = widgets.Button(description='プロット(x,y)')
    button_add_line_y = widgets.Button(description='プロット(y)')
    button_add_scatter_xy = widgets.Button(description='散布図(x,y)')
    button_add_scatter_y = widgets.Button(description='散布図(y)')
    button_show_fig = widgets.Button(description='描写')
    filechooser = FileChooser('./')
    button_load_as_x_npy = widgets.Button(description='x.npy読込')
    button_load_as_y_npy = widgets.Button(description='y.npy読込')
    button_load_as_x_txt = widgets.Button(description='x.txt読込')
    button_load_as_y_txt = widgets.Button(description='y.txt読込')
    button_load_csv = widgets.Button(description='csv読込')
    button_load_csv_dialog   = widgets.Button(description='csv読込ダイアログ')
    button_load_multiple_csv_dialog = widgets.Button(description='複数csv読込ダイアログ')
    button_save_csv = widgets.Button(description='csv保存')
    button_edit_df_dialog = widgets.Button(description='df編集ダイアログ')
    button_plotly_scatter_dialog = widgets.Button(description='scatterダイアログ')
    button_plotly_scatter_3d_dialog = widgets.Button(description='scatter3Dダイアログ')
    button_plotly_line_dialog = widgets.Button(description='lineダイアログ')
    button_plotly_bar_dialog = widgets.Button(description='barダイアログ')
    button_plotly_histogram_dialog = widgets.Button(description='histogramダイアログ')
    button_plotly_icicle_dialog = widgets.Button(description='icicleダイアログ')
    button_plotly_sunburst_dialog = widgets.Button(description='sunburstダイアログ')
    button_plotly_treemap_dialog = widgets.Button(description='treemapダイアログ')
    button_plotly_save_fig = widgets.Button(description='plotly_save_fig')
    button_pygwalker_dialog = widgets.Button(description='pygwalkerダイアログ')
    button_yf_query_dialog = widgets.Button(description='yf_queryダイアログ')
    button_yf_download_dialog = widgets.Button(description='yf_downloadダイアログ')
    button_yf_sector_dialog = widgets.Button(description='yf_sectorダイアログ')
    button_yf_ticker_dialog = widgets.Button(description='yf_tickerダイアログ')
    button_calc_return_dialog = widgets.Button(description='calc_returnダイアログ')



    
    button_save_fig_pdf = widgets.Button(description='pdf保存')
    button_save_fig_png = widgets.Button(description='png保存')

    button_input_field = widgets.Button(description='変数反映')

    text_comment = widgets.Text(value='',description='comment')
    text_xlabel = widgets.Text(value='',description='xlabel')
    text_ylabel = widgets.Text(value='',description='ylabel')
    text_title = widgets.Text(value='',description='title')
    text_xlim_high = widgets.Combobox(value='None',description='xlim_high',options=['None', '2020/1/1 00:00:00'],layout= widgets.Layout())
    text_xlim_low = widgets.Combobox(value='None',description='xlim_low',options=['None', '2019/1/1 00:00:00'],layout= widgets.Layout())
    text_ylim_high = widgets.Combobox(value='None',description='ylim_high',options=['None'],layout= widgets.Layout())
    text_ylim_low = widgets.Combobox(value='None',description='ylim_low',options=['None'],layout= widgets.Layout())
    text_delimiter = widgets.Combobox(value=',', description='delimiter',ensure_option=False, options=[',','\\t',' '])
    dropdown_marker = widgets.Dropdown(value=None ,description='marker', options=[(v,k) for k, v in mpl.markers.MarkerStyle.markers.items()])
    text_marker_size = widgets.Combobox(value='None', description='marker_size', options=['None'])
    text_linewidth = widgets.Combobox(value='None',description='linewidth', options=['None'])
    color_enable = widgets.Dropdown(value=False, description='color',options=[('color_pick',True), ('None',False),])
    color_picker = widgets.ColorPicker(concise=False,value='blue',disabled=False)
    widgets.jslink((color_enable,'index'),(color_picker,'disabled'))
    checkbox_withColumn = widgets.Checkbox(description='withColumn',value=True)

    

    button_x_transpose = widgets.Button(description='x転置')
    button_y_transpose = widgets.Button(description='y転置')


    output = widgets.Output()
    def wrapped_func_factory(func):
        def new_func(ui_element):
            with output:
                print(f"exec func {func.__name__}")
                func()
                print(f"complete {func.__name__}")
        return new_func
    button_clear_output.on_click(lambda button: output.clear_output(wait=False))
    button_init_fig.on_click(wrapped_func_factory(init_fig))
    button_set_label_title.on_click(wrapped_func_factory(set_label_title))
    button_add_line_xy.on_click(wrapped_func_factory(add_line_xy))
    button_add_line_y.on_click(wrapped_func_factory(add_line_y))
    button_add_scatter_xy.on_click(wrapped_func_factory(add_scatter_xy))
    button_add_scatter_y.on_click(wrapped_func_factory(add_scatter_y))
    button_show_fig.on_click(wrapped_func_factory(show_fig))
    button_set_axis_lim.on_click(wrapped_func_factory(set_axis_lim))
    button_set_autoscale.on_click(wrapped_func_factory(set_autoscale))

    def get_from_state_select(key, options=None, default=None):
        select = state_select.get(key, default)
        select = select if (options is None) or (select in options) else default
        return select

    def update_state_select_factory(key):
        def update(change):
            state_select[key] = change['new']
        return update

    def load_npy_factory(variable_name):
        def load_npy():
            choosed_file_path = filechooser.selected
            global x, y
            if variable_name == 'x':
                x = np.load(choosed_file_path)
                print(f'x.shape={x.shape}')
            elif variable_name == 'y':
                y = np.load(choosed_file_path)
                print(f'y.shape={y.shape}')
        return load_npy
    def load_txt_factory(variable_name):
        def load_txt():
            choosed_file_path = filechooser.selected
            global x, y
            if variable_name == 'x':
                x = np.loadtxt(choosed_file_path, delimiter=delimiter)
                print(f'x.shape={x.shape}')
            elif variable_name == 'y':
                y = np.loadtxt(choosed_file_path, delimiter=delimiter)
                print(f'y.shape={y.shape}')
        return load_txt
    def transpose_factory(variable_name):
        def transpose():
            global x,y
            if variable_name == 'x':
                x = x.T
                print(f'x.shape={x.shape}')
            elif variable_name == 'y':
                y = y.T
                print(f'y.shape={y.shape}')
        return transpose
    def load_csv():
        choosed_file_path = filechooser.selected
        global withColumn
        global df
        df = pl.read_csv(choosed_file_path, separator=delimiter, has_header=withColumn, infer_schema=True, infer_schema_length=None)

    def load_csv_dialog():
        global x,y
        global withColumn
        global df
        load_csv()
        selection = widgets.SelectMultiple(options=df.columns,description='columns')
        button_load_as_x = widgets.Button(description='x読込')
        button_load_as_y = widgets.Button(description='y読込')
        button_load_as_x_as_datetime = widgets.Button(description='x.datetime読込')
        def load_as_x():
            global x
            x = df[list(selection.value)].to_numpy()
            print(f'x.shape={x.shape}')
        def load_as_y():
            global y
            y = df[list(selection.value)].to_numpy()
            print(f'y.shape={y.shape}')
        def load_as_x_as_datetime():
            global x
            assert len(selection.value) == 1
            x = df.select(pl.col(selection.value[0]).str.to_datetime()).to_numpy()
            print(f'x.shape={x.shape}')
        button_load_as_x.on_click(wrapped_func_factory(load_as_x))
        button_load_as_y.on_click(wrapped_func_factory(load_as_y))
        button_load_as_x_as_datetime.on_click(wrapped_func_factory(load_as_x_as_datetime))
        display(
            widgets.VBox([
                widgets.HBox([selection,button_load_as_x,button_load_as_y,button_load_as_x_as_datetime]),
                df
            ])
        )
    

    def load_multiple_csv_dialog():
        global df
        dir_path = Path(filechooser.selected)
        assert(dir_path.is_dir())

        files = list(dir_path.glob('*.csv'))

        files_stem = [f.stem for f in files]
        conv_func = lambda x : {each.split('=')[0]:each.split('=')[1] for each in x.split(',') if '=' in each}

        generated_items = [conv_func(s) for s in files_stem]
        df_columns = pl.DataFrame(generated_items).columns

        dfs = []
        for file in files:
            df_load = pl.read_csv(file, separator=delimiter, has_header=withColumn, infer_schema=True, infer_schema_length=None)
            filename = file.stem
            dict_category = conv_func(filename)

            df_each = df_load.select(
                ([pl.lit(filename).alias('filename')] if len(df_columns)==0 else []) + [pl.lit(dict_category.get(column, '')).alias(column) for column in df_columns] + [pl.all()] 
            )
            dfs.append(df_each)
        df = pl.concat(dfs, how="diagonal_relaxed")
        display(df)

    def save_csv():
        global df
        global comment
        df.write_csv(f'{comment}.csv')

    def edit_df_dialog():
        global df
        button_change_col_name = widgets.Button(description='col名前変更')
        button_extract_col = widgets.Button(description='col抽出')
        button_unpivot = widgets.Button(description='colカテゴリカル化')
        button_pivot = widgets.Button(description='pivot化')
        button_cast_to_float = widgets.Button(description='col float化')
        button_cast_to_string = widgets.Button(description='col string化')
        button_cast_to_date = widgets.Button(description='col date化')
        tabs_col = widgets.TagsInput(value=df.columns)

        def change_col_name():
            global df
            df.columns = tabs_col.value
            display(df)
        
        def extract_col():
            global df
            df = df[tabs_col.value]
            display(df)

        def unpivot():
            global df
            df = df.unpivot(tabs_col.value, index=[x for x in df.columns if x not in tabs_col.value], variable_name='category', value_name='value')
            display(df)

        def pivot():
            global df
            assert len(tabs_col.value) == 2
            category=tabs_col.value[0]
            value=tabs_col.value[1]
            df = df.pivot(values=value, index=[x for x in df.columns if x not in [category, value]], columns=category)
            display(df)

        def cast_to_float():
            global df
            df = df.with_columns(
                [pl.col(tabs_col.value).cast(pl.datatypes.Float64, strict=False)]
            )
            display(df)

        def cast_to_string():
            global df
            df = df.with_columns(
                [pl.col(tabs_col.value).cast(pl.String)]
            )
            display(df)

        def cast_to_date():
            global df
            df = df.with_columns(
                [pl.col(tabs_col.value).str.to_date()]
            )
            display(df)

        
        button_change_col_name.on_click(wrapped_func_factory(change_col_name))
        button_extract_col.on_click(wrapped_func_factory(extract_col))
        button_unpivot.on_click(wrapped_func_factory(unpivot))
        button_pivot.on_click(wrapped_func_factory(pivot))
        button_cast_to_float.on_click(wrapped_func_factory(cast_to_float))
        button_cast_to_string.on_click(wrapped_func_factory(cast_to_string))
        button_cast_to_date.on_click(wrapped_func_factory(cast_to_date))

        display(
            widgets.VBox([
                widgets.HBox([button_change_col_name, button_extract_col, button_unpivot, button_pivot, button_cast_to_float, button_cast_to_string, button_cast_to_date]),
                tabs_col
            ])
        )



    def plotly_scatter_dialog():
        global df
        options = [None] + df.columns
        x_selection = widgets.Select(options=options,description='x', value=get_from_state_select('x', options))
        y_selection = widgets.Select(options=options,description='y', value=get_from_state_select('y', options))
        color_selection = widgets.Select(options=options,description='color', value=get_from_state_select('color', options))
        size_selection = widgets.Select(options=options,description='size', value=get_from_state_select('size', options))
        symbol_selection = widgets.Select(options=options,description='symbol', value=get_from_state_select('symbol', options))
        animation_frame_selection = widgets.Select(options=options,description='animation_frame', value=get_from_state_select('animation_frame', options))
        facet_col_selection = widgets.Select(options=options,description='facet_col', value=get_from_state_select('facet_col', options))
        facet_row_selection = widgets.Select(options=options,description='facet_row', value=get_from_state_select('facet_row', options))
        button_draw = widgets.Button(description='draw')

        x_selection.observe(update_state_select_factory('x'), names='value')
        y_selection.observe(update_state_select_factory('y'), names='value')
        color_selection.observe(update_state_select_factory('color'), names='value')
        size_selection.observe(update_state_select_factory('size'), names='value')
        symbol_selection.observe(update_state_select_factory('symbol'), names='value')
        animation_frame_selection.observe(update_state_select_factory('animation_frame'), names='value')
        facet_col_selection.observe(update_state_select_factory('facet_col'), names='value')
        facet_row_selection.observe(update_state_select_factory('facet_row'), names='value')

        def draw():
            global fig
            fig = px.scatter(
                data_frame=df,
                x = x_selection.value,
                y = y_selection.value,
                size = size_selection.value,
                symbol= symbol_selection.value,
                color = color_selection.value,
                facet_col = facet_col_selection.value,
                facet_row = facet_row_selection.value,
                animation_frame = animation_frame_selection.value
            )
            fig.show()

        button_draw.on_click(wrapped_func_factory(draw))

        display(
            widgets.VBox([
                widgets.HBox([x_selection, y_selection,color_selection]),
                widgets.HBox([size_selection,symbol_selection]),
                widgets.HBox([facet_col_selection, facet_row_selection, animation_frame_selection]),
                widgets.HBox([button_draw]),
            ])
        )

    def plotly_scatter_3d_dialog():
        global df
        options = [None] + df.columns
        x_selection = widgets.Select(options=[None] + df.columns,description='x', value=get_from_state_select('x', options))
        y_selection = widgets.Select(options=[None] + df.columns,description='y', value=get_from_state_select('y', options))
        z_selection = widgets.Select(options=options,description='z', value=get_from_state_select('z', options))
        color_selection = widgets.Select(options=options,description='color', value=get_from_state_select('color', options))
        size_selection = widgets.Select(options=options,description='size', value=get_from_state_select('size', options))
        symbol_selection = widgets.Select(options=options,description='symbol', value=get_from_state_select('symbol', options))
        animation_frame_selection = widgets.Select(options=options,description='animation_frame', value=get_from_state_select('animation_frame', options))
        button_draw = widgets.Button(description='draw')

        x_selection.observe(update_state_select_factory('x'), names='value')
        y_selection.observe(update_state_select_factory('y'), names='value')
        z_selection.observe(update_state_select_factory('z'), names='value')
        color_selection.observe(update_state_select_factory('color'), names='value')
        size_selection.observe(update_state_select_factory('size'), names='value')
        symbol_selection.observe(update_state_select_factory('symbol'), names='value')
        animation_frame_selection.observe(update_state_select_factory('animation_frame'), names='value')

        def draw():
            global fig
            fig = px.scatter_3d(
                data_frame=df,
                x = x_selection.value,
                y = y_selection.value,
                z = z_selection.value,
                color = color_selection.value,
                size = size_selection.value,
                symbol = symbol_selection.value,
                animation_frame = animation_frame_selection.value
            )
            fig.show()

        button_draw.on_click(wrapped_func_factory(draw))

        display(
            widgets.VBox([
                widgets.HBox([x_selection, y_selection, z_selection, color_selection]),
                widgets.HBox([size_selection, symbol_selection, animation_frame_selection]),
                widgets.HBox([button_draw])
            ])
        )
    
    def plotly_line_dialog():
        global df
        options = [None] + df.columns
        x_selection = widgets.Select(options=options,description='x', value=get_from_state_select('x', options))
        y_selection = widgets.Select(options=options,description='y', value=get_from_state_select('y', options))
        color_selection = widgets.Select(options=options,description='color', value=get_from_state_select('color', options))
        line_group_selection = widgets.Select(options=options,description='line_group', value=get_from_state_select('line_group', options))
        line_dash_selection = widgets.Select(options=options,description='line_dash', value=get_from_state_select('line_dash', options))
        animation_frame_selection = widgets.Select(options=options,description='animation_frame', value=get_from_state_select('animation_frame', options))
        facet_col_selection = widgets.Select(options=options,description='facet_col', value=get_from_state_select('facet_col', options))
        facet_row_selection = widgets.Select(options=options,description='facet_row', value=get_from_state_select('facet_row', options))
        symbol_selection = widgets.Select(options=options,description='symbol', value=get_from_state_select('symbol', options))

        x_selection.observe(update_state_select_factory('x'), names='value')
        y_selection.observe(update_state_select_factory('y'), names='value')
        color_selection.observe(update_state_select_factory('color'), names='value')
        line_group_selection.observe(update_state_select_factory('line_group'), names='value')
        line_dash_selection.observe(update_state_select_factory('line_dash'), names='value')
        animation_frame_selection.observe(update_state_select_factory('animation_frame'), names='value')
        facet_col_selection.observe(update_state_select_factory('facet_col'), names='value')
        facet_row_selection.observe(update_state_select_factory('facet_row'), names='value')
        symbol_selection.observe(update_state_select_factory('symbol'), names='value')

        button_draw = widgets.Button(description='draw')

        def draw():
            global fig
            fig = px.line(
                data_frame=df,
                x = x_selection.value,
                y = y_selection.value,
                line_group = line_group_selection.value,
                line_dash = line_dash_selection.value,
                facet_col = facet_col_selection.value,
                facet_row = facet_row_selection.value,
                animation_frame = animation_frame_selection.value,
                color = color_selection.value
            )
            fig.show()

        button_draw.on_click(wrapped_func_factory(draw))
        display(
            widgets.VBox([
                widgets.HBox([x_selection, y_selection,color_selection]),
                widgets.HBox([line_group_selection,line_dash_selection,symbol_selection]),
                widgets.HBox([facet_col_selection, facet_row_selection, animation_frame_selection]),
                widgets.HBox([button_draw]),
            ])
        )

    def plotly_bar_dialog():
        global df
        options = [None] + df.columns
        x_selection = widgets.Select(options=options,description='x', value=get_from_state_select('x', options))
        y_selection = widgets.Select(options=options,description='y', value=get_from_state_select('y', options))
        color_selection = widgets.Select(options=options,description='color', value=get_from_state_select('color', options))
        facet_col_selection = widgets.Select(options=options,description='facet_col', value=get_from_state_select('facet_col', options))
        facet_row_selection = widgets.Select(options=options,description='facet_row', value=get_from_state_select('facet_row', options))
        animation_frame_selection = widgets.Select(options=options,description='animation_frame', value=get_from_state_select('animation_frame', options))
        pattern_shape_selection = widgets.Select(options=options,description='pattern_shape', value=get_from_state_select('pattern_shape', options))
        
        barmode_selection = widgets.Select(options=['group','relative','overlay','stack'],description='barmode', value=get_from_state_select('barmode', ['group','relative','overlay','stack']))
        
        x_selection.observe(update_state_select_factory('x'), names='value')
        y_selection.observe(update_state_select_factory('y'), names='value')
        color_selection.observe(update_state_select_factory('color'), names='value')
        facet_col_selection.observe(update_state_select_factory('facet_col'), names='value')
        facet_row_selection.observe(update_state_select_factory('facet_row'), names='value')
        animation_frame_selection.observe(update_state_select_factory('animation_frame'), names='value')
        pattern_shape_selection.observe(update_state_select_factory('pattern_shape'), names='value')
        barmode_selection.observe(update_state_select_factory('barmode'), names='value')
        
        button_draw = widgets.Button(description='draw')

        def draw():
            global fig
            fig = px.bar(
                data_frame=df,
                x = x_selection.value,
                y = y_selection.value,
                color = color_selection.value,
                facet_col = facet_col_selection.value,
                facet_row = facet_row_selection.value,
                pattern_shape = pattern_shape_selection.value,
                animation_frame = animation_frame_selection.value,
                barmode = barmode_selection.value
            )
            fig.show()

        button_draw.on_click(wrapped_func_factory(draw))
        display(
            widgets.VBox([
                widgets.HBox([x_selection, y_selection,color_selection, pattern_shape_selection]),
                widgets.HBox([facet_col_selection, facet_row_selection, animation_frame_selection, barmode_selection]),
                widgets.HBox([button_draw]),
            ])
        )

    def plotly_histogram_dialog():
        global df
        options = [None] + df.columns
        x_selection = widgets.Select(options=options,description='x', value=get_from_state_select('x', options))
        y_selection = widgets.Select(options=options,description='y', value=get_from_state_select('y', options))
        color_selection = widgets.Select(options=options,description='color', value=get_from_state_select('color', options))
        pattern_shape_selection = widgets.Select(options=options,description='pattern_shape', value=get_from_state_select('pattern_shape', options))
        facet_col_selection = widgets.Select(options=options,description='facet_col', value=get_from_state_select('facet_col', options))
        facet_row_selection = widgets.Select(options=options,description='facet_row', value=get_from_state_select('facet_row', options))
        animation_frame_selection = widgets.Select(options=options,description='animation_frame', value=get_from_state_select('animation_frame', options))
        
        x_selection.observe(update_state_select_factory('x'), names='value')
        y_selection.observe(update_state_select_factory('y'), names='value')
        color_selection.observe(update_state_select_factory('color'), names='value')
        pattern_shape_selection.observe(update_state_select_factory('pattern_shape'), names='value')
        facet_col_selection.observe(update_state_select_factory('facet_col'), names='value')
        facet_row_selection.observe(update_state_select_factory('facet_row'), names='value')
        animation_frame_selection.observe(update_state_select_factory('animation_frame'), names='value')

        
        button_draw = widgets.Button(description='draw')

        def draw():
            global fig
            fig = px.histogram(
                data_frame=df,
                x = x_selection.value,
                y = y_selection.value,
                color = color_selection.value,
                pattern_shape = pattern_shape_selection.value,
                facet_col = facet_col_selection.value,
                facet_row = facet_row_selection.value,
                animation_frame = animation_frame_selection.value,
                barmode = 'group'
            )
            fig.show()

        button_draw.on_click(wrapped_func_factory(draw))
        display(
            widgets.VBox([
                widgets.HBox([x_selection, y_selection,color_selection, pattern_shape_selection]),
                widgets.HBox([facet_col_selection, facet_row_selection, animation_frame_selection]),
                widgets.HBox([button_draw]),
            ])
        )

    def plotly_icicle_dialog():
        global df
        options = [None] + df.columns
        values_selection = widgets.Select(options=options,description='values', value=get_from_state_select('values', options))
        color_selection = widgets.Select(options=options,description='color', value=get_from_state_select('color', options))
        path1_selection = widgets.Select(options=options,description='path1', value=get_from_state_select('path1', options))
        path2_selection = widgets.Select(options=options,description='path2', value=get_from_state_select('path2', options))
        path3_selection = widgets.Select(options=options,description='path3', value=get_from_state_select('path3', options))
        animation_frame_selection = widgets.Select(options=options,description='animation_frame', value=get_from_state_select('animation_frame', options))
        
        values_selection.observe(update_state_select_factory('values'), names='value')
        color_selection.observe(update_state_select_factory('color'), names='value')
        path1_selection.observe(update_state_select_factory('path1'), names='value')
        path2_selection.observe(update_state_select_factory('path2'), names='value')
        path3_selection.observe(update_state_select_factory('path3'), names='value')
        animation_frame_selection.observe(update_state_select_factory('animation_frame'), names='value')
        
        button_draw = widgets.Button(description='draw')
        button_draw_animation = widgets.Button(description='draw_anime')

        def draw_common(df):
            global fig
            path_source = [path1_selection, path2_selection, path3_selection]
            path = [px.Constant('total')] + [selection.value for selection in path_source if selection.value != None]
            
            fig = px.icicle(
                data_frame=df,
                values = values_selection.value,
                color = color_selection.value,
                path = path
            )

        def draw():
            global fig
            draw_common(df)
            fig.show()

        def draw_animation():
            global fig
            traces = []
            uniq = [v for v in df[animation_frame_selection.value].unique()]
            steps = []
            for i,v in enumerate(uniq):
                draw_common(df[df[animation_frame_selection.value]==v])
                fig.data[0].visible = False
                traces.append(fig.data[0])
                step = dict(
                    method="update",
                    args=[{"visible": [False] * len(uniq)}],
                    label=str(v)
                )
                step["args"][0]["visible"][i] = True
                steps.append(step)
            fig = go.Figure(data=traces)
            fig.data[0].visible = True
            sliders = [
                dict(
                    active=0,
                    currentvalue={"prefix": f"{animation_frame_selection.value} :"},
                    steps = steps
                )
            ]
            fig.update_layout(sliders=sliders)
            fig.show()

        button_draw.on_click(wrapped_func_factory(draw))
        button_draw_animation.on_click(wrapped_func_factory(draw_animation))
        display(
            widgets.VBox([
                widgets.HBox([values_selection, color_selection, animation_frame_selection]),
                widgets.HBox([path1_selection,path2_selection,path3_selection]),
                widgets.HBox([button_draw, button_draw_animation]),
            ])
        )

    def yf_query_dialog():
        global df

        set_region = set(yf.const.EQUITY_SCREENER_EQ_MAP['region']) # set(region)
        set_sector = yf.const.EQUITY_SCREENER_EQ_MAP['sector'] # set(sector)
        #set_industry = yf.const.EQUITY_SCREENER_EQ_MAP['industry'] # dict(sector -> set(industry))

        options_region = list(set_region)
        options_sector = [None] + list(set_sector)

        region_selection = widgets.Select(options=options_region,description='region', value=get_from_state_select('region', options_region))
        sector_selection = widgets.Select(options=options_sector,description='sector', value=get_from_state_select('sector', options_sector))
        button_fetch = widgets.Button(description='fetch')

        region_selection.observe(update_state_select_factory('region'), names='value')
        sector_selection.observe(update_state_select_factory('sector'), names='value')

        def fetch():
            global df
            q = yf.EquityQuery('eq', ['region', region_selection.value])
            if sector_selection.value != None:
                q_sector = yf.EquityQuery('eq', ['sector', sector_selection.value])
                q = yf.EquityQuery('and',[q,q_sector])
            
            result =fetch_yf_query(q, size=250)
            df = pl.DataFrame(result).drop("corporateActions")

        button_fetch.on_click(wrapped_func_factory(fetch))
        display(
            widgets.VBox([
                widgets.HBox([region_selection, sector_selection]),
                button_fetch
            ])
        )

    def yf_download_dialog():
        global df
        tickers_text = widgets.Text(description='tickers', value=get_from_state_select('tickers'))
        period_text = widgets.Text(description='period', value=get_from_state_select('period', default='5y'), placeholder='5y')
        start_text = widgets.Text(description='start', value=get_from_state_select('start'), placeholder='2020-01-01')
        end_text = widgets.Text(description='end', value=get_from_state_select('end'),placeholder='2025-01-01')
        options_interval = ["1m","5m","30m","60m","90m","1h","1d","1wk","1mo","1y"]
        interval_selection = widgets.Select(options=options_interval, description='interval', value=get_from_state_select('interval', options_interval, default='1d'))
        
        tickers_text.observe(update_state_select_factory('tickers'), names='value')
        period_text.observe(update_state_select_factory('period'), names='value')
        start_text.observe(update_state_select_factory('start'), names='value')
        end_text.observe(update_state_select_factory('end'), names='value')
        interval_selection.observe(update_state_select_factory('interval'), names='value')

        
        button_download = widgets.Button(description='download')
        
        def download():
            global df
            tickers = tickers_text.value.split(',')
            period = None if period_text.value == '' else period_text.value
            end = None if end_text.value == '' else end_text.value
            start = None if start_text.value == '' else start_text.value

            df = yf_download(tickers,period=period, start=start, end=end, interval=interval_selection.value)
            display(df)

        button_download.on_click(wrapped_func_factory(download))
        display(
            widgets.VBox([
                widgets.HBox([tickers_text]),
                widgets.HBox([period_text, start_text, end_text]),
                widgets.HBox([interval_selection]),
                button_download
            ])
        )

    def calc_return_dialog():
        global df
        options_on = df.columns
        on_selection = widgets.Select(options=options_on,description='calc_return_on', value=get_from_state_select('calc_return_on', options_on,default="Close"))

        on_selection.observe(update_state_select_factory('calc_return_on'), names='value')

        button_go = widgets.Button(description='go')

        def go():
            global df
            calc_return_on = on_selection.value
            df = calc_return(df, on=calc_return_on)
            display(df)

        button_go.on_click(wrapped_func_factory(go))
        display(
            widgets.VBox([
                widgets.HBox([on_selection]),
                button_go
            ])
        )

        

    def yf_sector_dialog():
        options = ['basic-materials','communication-services','consumer-cyclical','consumer-defensive','energy','financial-services','healthcare','industrials','real-estate','technology','utilities']
        sector_selection = widgets.Select(options=options, description='sector', value=get_from_state_select('sector', options=options))
        sector_selection.observe(update_state_select_factory('sector'), names='value')

        button_go = widgets.Button(description='go')

        def go():
            sector_value = sector_selection.value
            yf_sector(sector_value)
        
        button_go.on_click(wrapped_func_factory(go))
        display(
            widgets.VBox([
                widgets.HBox([sector_selection]),
                button_go
            ])
        )

    def yf_ticker_dialog():
        ticker_text = widgets.Text(description='ticker', value=get_from_state_select('ticker'))
        ticker_text.observe(update_state_select_factory('ticker'), names='value')
        
        ticker_text.observe(update_state_select_factory('ticker'), names='value')

        button_go = widgets.Button(description='go')

        def go():
            ticker = ticker_text.value
            yf_ticker(ticker)
        
        button_go.on_click(wrapped_func_factory(go))
        display(
            widgets.VBox([
                widgets.HBox([ticker_text]),
                button_go
            ])
        )


    def plotly_treemap_dialog():
        global df
        options = [None] + df.columns
        values_selection = widgets.Select(options=options,description='values', value=get_from_state_select('values', options))
        color_selection = widgets.Select(options=options,description='color', value=get_from_state_select('color', options))
        path1_selection = widgets.Select(options=options,description='path1', value=get_from_state_select('path1', options))
        path2_selection = widgets.Select(options=options,description='path2', value=get_from_state_select('path2', options))
        path3_selection = widgets.Select(options=options,description='path3', value=get_from_state_select('path3', options))
        animation_frame_selection = widgets.Select(options=options,description='animation_frame', value=get_from_state_select('animation_frame', options))
        
        values_selection.observe(update_state_select_factory('values'), names='value')
        color_selection.observe(update_state_select_factory('color'), names='value')
        path1_selection.observe(update_state_select_factory('path1'), names='value')
        path2_selection.observe(update_state_select_factory('path2'), names='value')
        path3_selection.observe(update_state_select_factory('path3'), names='value')
        animation_frame_selection.observe(update_state_select_factory('animation_frame'), names='value')
        
        button_draw = widgets.Button(description='draw')
        button_draw_animation = widgets.Button(description='draw_anime')

        def draw_common(df):
            global fig
            path_source = [path1_selection, path2_selection, path3_selection]
            path = [px.Constant('total')] + [selection.value for selection in path_source if selection.value != None]
            
            fig = px.treemap(
                data_frame=df,
                values = values_selection.value,
                color = color_selection.value,
                path = path
            )

        def draw():
            global fig
            draw_common(df)
            fig.show()

        def draw_animation():
            global fig
            traces = []
            uniq = [v for v in df[animation_frame_selection.value].unique()]
            steps = []
            for i,v in enumerate(uniq):
                draw_common(df.filter(pl.col(animation_frame_selection.value) == v))
                fig.data[0].visible = False
                traces.append(fig.data[0])
                step = dict(
                    method="update",
                    args=[{"visible": [False] * len(uniq)}],
                    label=str(v)
                )
                step["args"][0]["visible"][i] = True
                steps.append(step)
            fig = go.Figure(data=traces)
            fig.data[0].visible = True
            sliders = [
                dict(
                    active=0,
                    currentvalue={"prefix": f"{animation_frame_selection.value} :"},
                    steps = steps
                )
            ]
            fig.update_layout(sliders=sliders)
            fig.show()
        
        button_draw.on_click(wrapped_func_factory(draw))
        button_draw_animation.on_click(wrapped_func_factory(draw_animation))
        display(
            widgets.VBox([
                widgets.HBox([values_selection, color_selection, animation_frame_selection]),
                widgets.HBox([path1_selection,path2_selection,path3_selection]),
                widgets.HBox([button_draw, button_draw_animation]),
            ])
        )
    
    def plotly_sunburst_dialog():
        global df
        options = [None] + df.columns
        values_selection = widgets.Select(options=options,description='values', value=get_from_state_select('values', options))
        color_selection = widgets.Select(options=options,description='color', value=get_from_state_select('color', options))
        path1_selection = widgets.Select(options=options,description='path1', value=get_from_state_select('path1', options))
        path2_selection = widgets.Select(options=options,description='path2', value=get_from_state_select('path2', options))
        path3_selection = widgets.Select(options=options,description='path3', value=get_from_state_select('path3', options))
        animation_frame_selection = widgets.Select(options=options,description='animation_frame', value=get_from_state_select('animation_frame', options))
        
        values_selection.observe(update_state_select_factory('values'), names='value')
        color_selection.observe(update_state_select_factory('color'), names='value')
        path1_selection.observe(update_state_select_factory('path1'), names='value')
        path2_selection.observe(update_state_select_factory('path2'), names='value')
        path3_selection.observe(update_state_select_factory('path3'), names='value')
        animation_frame_selection.observe(update_state_select_factory('animation_frame'), names='value')
        
        button_draw = widgets.Button(description='draw')
        button_draw_animation = widgets.Button(description='draw_anime')

        def draw_common(df):
            global fig
            path_source = [path1_selection, path2_selection, path3_selection]
            path = [selection.value for selection in path_source if selection.value != None]
            
            fig = px.sunburst(
                data_frame=df,
                values = values_selection.value,
                color = color_selection.value,
                path = path
            )

        def draw():
            global fig
            draw_common(df)
            fig.show()

        def draw_animation():
            global fig
            traces = []
            uniq = [v for v in df[animation_frame_selection.value].unique()]
            steps = []
            for i,v in enumerate(uniq):
                draw_common(df.filter(pl.col(animation_frame_selection.value) == v))
                fig.data[0].visible = False
                traces.append(fig.data[0])
                step = dict(
                    method="update",
                    args=[{"visible": [False] * len(uniq)}],
                    label=str(v)
                )
                step["args"][0]["visible"][i] = True
                steps.append(step)
            fig = go.Figure(data=traces)
            fig.data[0].visible = True
            sliders = [
                dict(
                    active=0,
                    currentvalue={"prefix": f"{animation_frame_selection.value} :"},
                    steps = steps
                )
            ]
            fig.update_layout(sliders=sliders)
            fig.show()
        
        button_draw.on_click(wrapped_func_factory(draw))
        button_draw_animation.on_click(wrapped_func_factory(draw_animation))
        display(
            widgets.VBox([
                widgets.HBox([values_selection, color_selection, animation_frame_selection]),
                widgets.HBox([path1_selection,path2_selection,path3_selection]),
                widgets.HBox([button_draw, button_draw_animation]),
            ])
        )

    def plotly_save_fig():
        global fig
        global comment
        pio.write_html(fig, f"{comment}.html")
        print(f"save to {comment}.html")

    def pygwalker_dialog():
        global df
        filechooser_forspecs = FileChooser('./')
        filechooser_forspecs.filter_pattern = ['*.json']
        button_draw = widgets.Button(description='draw')


        def draw():
            global df
            selected_spec_file = filechooser_forspecs.selected
            spec = ''
            if selected_spec_file is not None:
                with open(selected_spec_file, 'r') as file:
                    spec = file.read()
            print(spec)
            pyg.walk(df, spec=spec, kernel_computation=True)
        
        button_draw.on_click(wrapped_func_factory(draw))
        display(
            widgets.VBox([
                widgets.HBox([filechooser_forspecs]),
                widgets.HBox([button_draw])
            ])
        )



    button_load_as_x_npy.on_click(wrapped_func_factory(load_npy_factory('x')))
    button_load_as_y_npy.on_click(wrapped_func_factory(load_npy_factory('y')))
    button_load_as_x_txt.on_click(wrapped_func_factory(load_txt_factory('x')))
    button_load_as_y_txt.on_click(wrapped_func_factory(load_txt_factory('y')))
    button_load_csv_dialog.on_click(wrapped_func_factory(load_csv_dialog))
    button_load_csv.on_click(wrapped_func_factory(load_csv))
    button_load_multiple_csv_dialog.on_click(wrapped_func_factory(load_multiple_csv_dialog))
    button_save_csv.on_click(wrapped_func_factory(save_csv))
    button_edit_df_dialog.on_click(wrapped_func_factory(edit_df_dialog))
    button_plotly_scatter_dialog.on_click(wrapped_func_factory(plotly_scatter_dialog))
    button_plotly_scatter_3d_dialog.on_click(wrapped_func_factory(plotly_scatter_3d_dialog))
    button_plotly_bar_dialog.on_click(wrapped_func_factory(plotly_bar_dialog))
    button_plotly_histogram_dialog.on_click(wrapped_func_factory(plotly_histogram_dialog))
    button_plotly_line_dialog.on_click(wrapped_func_factory(plotly_line_dialog))
    button_plotly_icicle_dialog.on_click(wrapped_func_factory(plotly_icicle_dialog))
    button_plotly_sunburst_dialog.on_click(wrapped_func_factory(plotly_sunburst_dialog))
    button_plotly_treemap_dialog.on_click(wrapped_func_factory(plotly_treemap_dialog))
    button_plotly_save_fig.on_click(wrapped_func_factory(plotly_save_fig))
    button_pygwalker_dialog.on_click(wrapped_func_factory(pygwalker_dialog))
    button_yf_query_dialog.on_click(wrapped_func_factory(yf_query_dialog))
    button_yf_download_dialog.on_click(wrapped_func_factory(yf_download_dialog))
    button_yf_sector_dialog.on_click(wrapped_func_factory(yf_sector_dialog))
    button_yf_ticker_dialog.on_click(wrapped_func_factory(yf_ticker_dialog))
    button_calc_return_dialog.on_click(wrapped_func_factory(calc_return_dialog))

    button_save_fig_pdf.on_click(wrapped_func_factory(save_fig_pdf))
    button_save_fig_png.on_click(wrapped_func_factory(save_fig_png))
    button_x_transpose.on_click(wrapped_func_factory(transpose_factory('x')))
    button_y_transpose.on_click(wrapped_func_factory(transpose_factory('y')))



    def load_input_field():
        global comment
        global xlabel
        global ylabel
        global title
        global xlim_high, xlim_low
        global ylim_high, ylim_low
        global delimiter
        global marker, marker_size
        global linewidth
        global color
        global withColumn

        comment = text_comment.value
        xlabel = text_xlabel.value
        ylabel = text_ylabel.value
        title = text_title.value
        lims = [text_xlim_high.value, text_xlim_low.value, text_ylim_high.value, text_ylim_low.value]
        withColumn = checkbox_withColumn.value
        def convert_lim(string):
            from dateutil import parser
            if string in ['None','']:
                return None
            else:
                try: return float(string)
                except: pass
                try: return parser.parse(string)
                except Exception as e: print(e)
        def none_or_float(string):
            if string in ['None','']: return None
            else:
                try: return float(string)
                except Exception as e: print(e)
        xlim_high, xlim_low, ylim_high, ylim_low = [convert_lim(i) for i in lims]
        delimiter = text_delimiter.value
        marker = dropdown_marker.value
        marker_size = none_or_float(text_marker_size.value)
        linewidth = none_or_float(text_linewidth.value)
        color = color_picker.value if color_enable.value == True else None
        
    button_input_field.on_click(wrapped_func_factory(load_input_field))
    load_input_field()
    display(
        widgets.VBox([
            widgets.HBox([button_clear_output, button_init_fig, button_set_label_title ,button_set_axis_lim,button_set_autoscale , button_show_fig]),
            widgets.HBox([button_add_line_y,button_add_line_xy, button_add_scatter_y, button_add_scatter_xy]),
            widgets.HBox([filechooser,button_load_as_x_npy, button_load_as_y_npy]),
            widgets.HBox([button_load_as_x_txt,button_load_as_y_txt,button_load_csv,button_load_csv_dialog,button_save_fig_png,button_save_fig_pdf]),
            widgets.Accordion(children=[
                widgets.VBox([
                    widgets.HBox([text_comment]),
                    widgets.HBox([text_xlabel, text_ylabel, text_title ,button_input_field]),
                    widgets.HBox([text_xlim_low,text_xlim_high, text_ylim_low, text_ylim_high]),
                    widgets.HBox([text_delimiter,dropdown_marker,text_marker_size,text_linewidth]),
                    widgets.HBox([color_enable, color_picker, checkbox_withColumn])
                ])
            ],  _titles={0:'各種設定'}, selected_index=None),
            widgets.HBox([button_x_transpose,button_y_transpose, button_edit_df_dialog, button_load_multiple_csv_dialog, button_save_csv]),
            widgets.HBox([button_plotly_scatter_dialog, button_plotly_scatter_3d_dialog, button_plotly_line_dialog, button_plotly_icicle_dialog, button_plotly_sunburst_dialog]),
            widgets.HBox([button_plotly_treemap_dialog,button_plotly_bar_dialog, button_plotly_histogram_dialog, button_plotly_save_fig]),
            widgets.HBox([button_pygwalker_dialog]),
            widgets.HBox([button_yf_query_dialog, button_yf_download_dialog, button_yf_sector_dialog, button_yf_ticker_dialog]),
            widgets.HBox([button_calc_return_dialog]),
            output,
        ])
    )


In [4]:
simple_ui()

In [5]:
ticker_map = {
    "1615.T" : "NF・銀行業（東証33）ETF",
    "1617.T" : "NF・食品（TPX17）ETF",
    "1618.T" : "NF・エネルギー資源（TPX17）ETF",
    "1619.T" : "NF・建設・資材（TPX17）ETF",
    "1620.T" : "NF・素材・化学（TPX17）ETF",
    "1621.T" : "NF・医薬品（TPX17）ETF",
    "1622.T" : "NF・自動車・輸送機（TPX17）ETF",
    "1623.T" : "NF・鉄鋼・非鉄（TPX17）ETF",
    "1624.T" : "NF・機械（TPX17）ETF",
    "1625.T" : "NF・電機・精密（TPX17）ETF",
    "1626.T" : "NF・情報・サービス他（TPX17）ETF",
    "1627.T" : "NF・電力・ガス（TPX17）ETF",
    "1628.T" : "NF・運輸・物流（TPX17）ETF",
    "1629.T" : "NF・商社・卸売（TPX17）ETF",
    "1630.T" : "NF・小売（TPX17）ETF",
    "1631.T" : "NF・銀行（TPX17）ETF",
    "1632.T" : "NF・金融（TPX17）ETF",
    "1633.T" : "NF・不動産（TPX17）ETF",
}

In [20]:
df = yf_download(tickers=list(ticker_map.keys()), period="max")
df = calc_return(df)
df = df.drop_nulls("return")

In [69]:
df_monthly = (
    df
    .with_columns(
        pl.col("Date").dt.truncate("1mo").alias("Month")
    )
    .group_by(["Ticker", "Month"])
    .agg(
        pl.col("return").mean().alias("mean_return")
    )
    .sort(["Ticker", "Month"])
    .with_columns([
        pl.col("Ticker").replace(ticker_map)
    ])
)

In [71]:
import polars as pl
import plotly.graph_objects as go

# --- 1. ピボット（Ticker × Month の行列にする） ---
df_pivot = (
    df_monthly
    .pivot(
        values="mean_return",
        index="Ticker",
        on="Month"
    )
    .sort("Ticker")
)

# --- 2. 軸データ作成 ---
x = df_pivot.columns[1:]  # Month
y = df_pivot["Ticker"].to_list()
z = df_pivot.select(x).to_numpy()

# --- 3. ヒートマップ ---
fig = go.Figure(
    data=go.Heatmap(
        z=z,
        x=x,
        y=y,
        colorscale="RdBu",
        zmid=0  # リターンなので0を中心に
    )
)

fig.update_layout(
    title="Mean Return Heatmap (Ticker × Month)",
    xaxis_title="Month",
    yaxis_title="Ticker"
)
fig.update_layout(
    paper_bgcolor="#0b0f14",
    plot_bgcolor="#0b0f14",
    font=dict(color="#d0d0d0", family="Arial"),
    legend=dict(
        bgcolor="rgba(0,0,0,0)",
        font=dict(color="#d0d0d0")
    ),
    margin=dict(t=40, b=40, l=60, r=30)
)

fig.show()

In [60]:
import polars as pl
import numpy as np
import plotly.graph_objects as go


bbg_colorscale = [
    [0.0, "#8b0000"],   # 強い負（ダークレッド）
    [0.25, "#ff4d4d"],  # 弱い負
    [0.5, "#1a1a1a"],   # 0付近（ほぼ黒）
    [0.75, "#66ff66"],  # 弱い正
    [1.0, "#00cc00"]    # 強い正（グリーン）
]

# --- 1. Quarter作成 & ソート ---
df_q = (
    df
    .with_columns(
        pl.col("Date").dt.truncate("3mo").alias("Quarter"),
        pl.col("Ticker").replace(ticker_map)
    )
    .sort("Quarter")
)

quarters = df_q.select("Quarter").unique().sort("Quarter")["Quarter"].to_list()

# --- 2. Ticker順を固定（これが超重要） ---
tickers = (
    df_q.select("Ticker")
    .unique()
    .sort("Ticker")
    ["Ticker"]
    .to_list()
)

frames = []

# --- 3. 各Quarterごとに相関行列 ---
for q in quarters:
    df_sub = df_q.filter(pl.col("Quarter") == q)

    df_wide = (
        df_sub
        .pivot(
            values="return",
            index="Date",
            on="Ticker"
        )
        .sort("Date")
    )

    # --- 列順を固定（存在しないTickerはnullで埋める） ---
    df_wide = df_wide.select(
        ["Date"] + [t for t in tickers if t in df_wide.columns]
    )

    data = df_wide.drop("Date").to_numpy()

    # 欠損対応
    data = np.nan_to_num(data)

    # 相関
    corr = np.corrcoef(data, rowvar=False)

    frames.append(
        go.Frame(
            data=[
                go.Heatmap(
                    z=corr,
                    x=tickers,
                    y=tickers,
                    colorscale="RdBu",
                    zmid=0,
                    zmin=-1,
                    zmax=1,
                    
                )
            ],
            name=str(q)
        )
    )

# --- 4. 初期データ ---
fig = go.Figure(
    data=frames[0].data,
    frames=frames
)

# --- 5. スライダー ---
fig.update_layout(
    title="Quarterly Correlation Heatmap",
    xaxis_title="Ticker",
    yaxis_title="Ticker",
    yaxis_autorange="reversed",
    sliders=[
        {
            "steps": [
                {
                    "method": "animate",
                    "label": str(q),
                    "args": [[str(q)], {"mode": "immediate", "frame": {"duration": 0}}]
                }
                for q in quarters
            ]
        }
    ]
)
fig.update_xaxes(side="top", tickangle=-20)
fig.update_layout(
    paper_bgcolor="#0b0f14",
    plot_bgcolor="#0b0f14",
    font=dict(color="#d0d0d0", family="Arial"),
    legend=dict(
        bgcolor="rgba(0,0,0,0)",
        font=dict(color="#d0d0d0")
    ),
    margin=dict(t=40, b=40, l=60, r=30)
)

fig.update_xaxes(
    showgrid=True,
    gridcolor="#2a2f36",
    linecolor="#aaaaaa",
    zeroline=False
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="#2a2f36",
    linecolor="#1c1b1b",
    zeroline=False
)

fig.show()